# Feature Engineering

## What is Feature Engineering?

**Feature engineering** is the process of using domain knowledge to transform raw data into meaningful inputs (called *features*) for a machine learning model or statistical analysis.

Raw data on its own is rarely in a form that models can learn from directly. For example:
- A **timestamp** tells you when something happened — but your model cannot directly use `'2020-05-26 22:15:00'`. A more useful feature might be `days_before_deadline`.
- A **list of page visits** does not tell you much alone — but `total number of visits` or `time spent per assignment` are informative summaries.

In this lecture, we will practice feature engineering on a real dataset: **Canvas LMS log data** from an online programming course. Our goal is to build features that help explain why some students perform better on assignments than others.

---

### The three feature families we will explore today

| # | Feature family | Example | Intuition |
|---|---|---|---|
| 1 | **Count & time aggregates** | `totalCount`, `totalTime` | How much did the student engage overall? |
| 2 | **Temporal / day-based features** | `firstDay`, `nDays`, `nDaysFinalWeek` | *When* did the student work — did they cram or spread it out? |
| 3 | **Ratios** | `detailRatio` | What *proportion* of their time was spent on detail vs. overview pages? |

---

We continue from Week 4 where we computed `view_length` (capped at 30 minutes). Let's reload the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('default')

# ── Load the Canvas log data ──────────────────────────────────────────────────
# Each row is one page-view event: which student (userID) visited which
# assignment URL, and for how long (view_length, in seconds, capped at 1800).
log = pd.read_csv('log_assignments_viewlength_1800.csv')
print('Log shape:', log.shape)
print(log.head(10))

# ── Load the grades data ──────────────────────────────────────────────────────
# Each row is one student.  Columns A1–A5 are assignment grades (0–100).
# FinalScore is the overall course grade.
grades = pd.read_csv('grades_data.csv')
print('\nGrades shape:', grades.shape)
print(grades.head(10))

---
## Feature Family 1 — Count & Time Aggregates

The simplest features we can compute are **totals per student per assignment**:
- `totalCount` — how many times did the student visit any page for this assignment?
- `totalTime` — how many seconds in total did the student spend on those pages?

These are called **aggregation features**: we are collapsing many rows (individual page views) into one summary row per (student, assignment) pair.

> 💡 **Pandas pattern:** `groupby(['userID','assignment']).size()` counts rows per group.
> `.unstack()` pivots the `assignment` level into separate columns, giving one column per assignment.

In [ ]:
# ── Step 1: Count total page views per student per assignment ────────────────
#
# .groupby(['userID','assignment'])  →  form a group for every (student, assignment) pair
# .size()                            →  count how many rows (events) are in each group
# .unstack(fill_value=0)             →  pivot 'assignment' into columns; fill missing with 0
#   (a student who never visited a5 should have 0, not NaN)

grouped = log.groupby(['userID', 'assignment']).size().unstack(fill_value=0)

# Sort columns so they appear as a1, a2, a3 ... (not in arbitrary order)
grouped = grouped.reindex(sorted(grouped.columns), axis=1)

# Add a meaningful suffix so we remember what these columns contain
grouped.columns = [f"{col}totalCount" for col in grouped.columns]

# Reset index: makes userID a plain column instead of the DataFrame index
assignment_counts = grouped.reset_index()

print(f"Shape: {assignment_counts.shape}  →  {assignment_counts.shape[0]} students, "
      f"{assignment_counts.shape[1]-1} count features + userID")
assignment_counts.head()

Now let's do the same thing for **time**. Instead of counting rows, we *sum* the `view_length` column within each group.

> 💡 `groupby(...)[col].sum()` is a common pandas idiom for aggregating a specific numeric column.

In [ ]:
# ── Step 2: Sum view_length (seconds) per student per assignment ─────────────
#
# Same groupby structure as above, but instead of .size() we use
# ['view_length'].sum() to add up all the seconds in each group.

time_totals = (
    log.groupby(['userID', 'assignment'])['view_length']
       .sum()
       .unstack(fill_value=0)
)
time_totals = time_totals.reindex(sorted(time_totals.columns), axis=1)
time_totals.columns = [f"{col}totalTime" for col in time_totals.columns]
time_totals = time_totals.reset_index()

# ── Step 3: Merge counts and times into one 'assignments' DataFrame ───────────
#
# We now have two wide DataFrames with one row per student.
# pd.merge(..., on='userID', how='left') joins them by matching userID.
# 'left' means: keep all students from assignment_counts even if they
# somehow have no rows in time_totals.

assignments = pd.merge(assignment_counts, time_totals, on='userID', how='left')

print(f"Final shape: {assignments.shape}")
print("Columns:", assignments.columns.tolist())
assignments.head()

### Explore the new features

Before using features in a model, always **visualise their distribution**.
This helps you spot outliers, skew, or data quality issues that could mislead the model.

We'll look at:
1. A **box plot** of `totalTime` — what is the spread across assignments?
2. A **scatter plot** of `totalCount` vs `totalTime` — are they correlated?

In [ ]:
# ── Plot 1: Distribution of totalTime per assignment (box plot) ──────────────
#
# sns.boxplot shows median, IQR, and outliers for each assignment side-by-side.
# .filter(like='totalTime') selects only the columns whose name contains 'totalTime'.
# orient='h' makes horizontal boxes — easier to read long assignment names.

plt.figure(figsize=(12, 5))
sns.boxplot(data=assignments.filter(like='totalTime'), orient='h', palette='pastel')
plt.title('Distribution of Total Time per Assignment', fontsize=14)
plt.xlabel('Total Time (seconds)  |  3 600 s = 1 hour')
plt.ylabel('Assignment')
plt.tight_layout()
plt.show()

# ── Plot 2: Scatter — totalCount vs totalTime for each assignment ─────────────
#
# If count and time are highly correlated they may be measuring the same thing.
# Here we plot them side-by-side to visually check.

count_cols = assignments.filter(like='totalCount').columns.tolist()
n = len(count_cols)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(count_cols):
    time_col = col.replace('totalCount', 'totalTime')
    axes[i].scatter(assignments[col], assignments[time_col], alpha=0.6, edgecolors='k', linewidths=0.3)
    axes[i].set_title(f"{col}  vs  {time_col}", fontsize=10)
    axes[i].set_xlabel('Count (# visits)')
    axes[i].set_ylabel('Time (seconds)')

# Hide unused subplot if fewer than 6 assignments
for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Total Visit Count vs. Total Time per Assignment', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# ── Quick sanity check: print max time to understand scale ───────────────────
max_time = assignments.filter(like='totalTime').max().max()
print(f"Max total time across all students/assignments: {max_time:.0f} s = {max_time/3600:.1f} hours")

### Merge with grades and check correlations

Now that we have our first engineered features, let's see if they actually **predict grades**.
We'll merge the feature DataFrame with the grades and compute Pearson correlations.

> 💡 Pearson correlation ranges from −1 (perfect negative) to +1 (perfect positive). 
> A value of 0 means no linear relationship. Correlations around ±0.2–0.4 are already interesting for behavioural data.

In [ ]:
# Merge the new features into the grades DataFrame
# After this, 'grades' has both the outcome columns (A1–A5) and our new features.
grades = pd.merge(grades, assignments, on='userID', how='left')

print(f"Grades DataFrame shape: {grades.shape}")
grades.head()

In [ ]:
# ── Compute correlation between each assignment grade and its totalTime ───────
#
# For each assignment i, we pick the grade column A{i} and the feature a{i}totalTime
# and compute the Pearson correlation between them.
# .corr() returns a 2×2 correlation matrix; .iloc[0,1] extracts the off-diagonal value.

correlation_results = {}

for i in range(1, 6):
    grade_col = f'A{i}'
    time_col  = f'a{i}totalTime'
    
    if grade_col in grades.columns and time_col in grades.columns:
        corr = grades[[grade_col, time_col]].corr().iloc[0, 1]
        correlation_results[grade_col] = round(corr, 3)

print("Correlation between assignment grade and total time on that assignment:")
for k, v in correlation_results.items():
    print(f"  {k} ↔ {k.lower()}totalTime :  r = {v}")

These are decent correlations for a behavioural feature.
A2 is the exception — we'd want to investigate the assignment content to understand why.

Let's also look at the **full correlation matrix** across all grade and feature columns to get a wider picture.

In [ ]:
# ── Full correlation heatmap ──────────────────────────────────────────────────
#
# grades.drop(columns=['userID'])  →  exclude the ID column (not a numeric feature)
# .corr()                          →  compute all pairwise Pearson correlations
# sns.heatmap with annot=True      →  print the correlation value inside each cell
# vmin/vmax=-1/1                   →  fix the colour scale to the full correlation range

correlation_matrix = grades.drop(columns=['userID']).corr()

plt.figure(figsize=(14, 11))
sns.heatmap(
    correlation_matrix,
    annot=True, fmt=".2f", cmap='coolwarm',
    square=True, cbar_kws={"shrink": .75},
    vmin=-1, vmax=1
)
plt.title('Correlation Matrix — Grades and Count/Time Features', fontsize=14)
plt.tight_layout()
plt.show()

print("\n📌 Observation: The correlation between totalCount and totalTime for the same"
      " assignment is in the 0.68–0.85 range.")
print("   This means they are related, but NOT redundant — each carries unique information.")

---
## Feature Family 2 — Temporal / Day-Based Features

Counts and time tell us *how much* a student engaged. 
But **when** they worked can be equally important:

| Feature | Question answered |
|---|---|
| `firstDay` | How early did they start (days before deadline)? |
| `secondDay` | When did they come back for the second session? |
| `nDays` | Over how many distinct days did they work? |
| `nDaysFinalWeek` | Did they cram everything into the last 7 days? |
| `nDaysAfterDeadline` | Did they access pages after the deadline (late submission risk)? |

> 💡 **Key idea:** A student who starts 14 days before the deadline and works a little each day
> has a very different pattern from one who starts the night before.
> These features give the model a way to capture that difference.

First, we need the **deadline dates** for each assignment.

In [ ]:
# ── Define assignment deadlines ───────────────────────────────────────────────
#
# We looked these up from the course records (or used DataWrangler to find the
# date with the highest access frequency just before activity dropped off).
# Timestamps include timezone offset (-07:00 = Pacific Daylight Time).

a1deadline = pd.to_datetime('2020-05-27 23:59:00-07:00')
a2deadline = pd.to_datetime('2020-06-15 23:59:00-07:00')
a3deadline = pd.to_datetime('2020-07-05 23:59:00-07:00')
a4deadline = pd.to_datetime('2020-07-15 23:59:00-07:00')
a5deadline = pd.to_datetime('2020-08-09 23:59:00-07:00')

# Store in a dictionary: assignment name → deadline Timestamp
# A dictionary is convenient here because we'll look up deadlines by assignment name
# while looping through the log data below.
deadlines = {
    'a1': a1deadline,
    'a2': a2deadline,
    'a3': a3deadline,
    'a4': a4deadline,
    'a5': a5deadline,
}

print("Deadlines loaded:")
for k, v in deadlines.items():
    print(f"  {k}: {v.date()}")

Now we loop over every (student, assignment) group in the log to compute the temporal features.

The key formula is:
```
days_before_deadline = (deadline.date() - access_date).days
```
A positive number means the student accessed the page **before** the deadline.
A negative number means they accessed it **after** — which might indicate late work or reviewing feedback.

In [ ]:
# ── Make sure start_time is a proper datetime with timezone info ─────────────
log['start_time'] = pd.to_datetime(log['start_time'], utc=True)

# Extract just the date part (drops the time of day) — we only care which *day* the visit happened
log['date'] = log['start_time'].dt.date

# ── Build temporal features for each (student, assignment) pair ───────────────
rows = []

for (user, assignment), group in log.groupby(['userID', 'assignment']):
    # Skip assignments we don't have a deadline for
    if assignment not in deadlines:
        continue
    
    deadline = deadlines[assignment]
    
    # Get sorted list of unique calendar days this student visited the assignment
    dates = sorted(set(group['date']))
    
    # Convert each date to "days before deadline"
    # Positive = before deadline, 0 = day of deadline, negative = after deadline
    days_before = [(deadline.date() - d).days for d in dates]
    
    # Days that fell within the final week (0–7 days before deadline)
    days_in_last_week = [d for d in dates if 0 <= (deadline.date() - d).days <= 7]
    
    rows.append({
        'userID':              user,
        'assignment':          assignment,

        # How early did the student first access the assignment?
        # Higher value = started earlier = less procrastination
        'firstDay':            days_before[0] if len(days_before) >= 1 else None,

        # When did they return for their second session?
        'secondDay':           days_before[1] if len(days_before) >= 2 else None,

        # Gap between first and second visit (in days)
        # A large gap might mean they read the brief, thought about it, then came back
        'firstSecondGap':      (days_before[0] - days_before[1]) if len(days_before) >= 2 else None,

        # Total number of distinct days the student worked on this assignment
        'nDays':               len(dates),

        # How many of those days were in the final week? (cramming indicator)
        'nDaysFinalWeek':      len(days_in_last_week),

        # Days accessed more than a week before the deadline (early-planner indicator)
        'nDaysBeforeFinalWeek': len([d for d in days_before if d > 7]),

        # Days accessed AFTER the deadline (late submission or reviewing feedback)
        'nDaysAfterDeadline':  len([d for d in days_before if d < 0]),
    })

activity_summary = pd.DataFrame(rows)
print(f"activity_summary shape: {activity_summary.shape}")
activity_summary.head(10)

### Reshaping the data: wide vs. long format

Right now `grades` is in **wide format** (one row per student, one column per assignment per feature).
`activity_summary` is in **long format** (one row per student–assignment pair).

To cleanly merge temporal features with grades, we'll convert everything to long format first.

> 💡 `pd.melt()` is the pandas function for converting from wide to long.
> It "unpivots" selected columns into rows.

In [ ]:
# ── Convert the wide 'grades' DataFrame to long format ───────────────────────
#
# Wide:  userID | a1totalCount | a2totalCount | ... | A1 | A2 | ...
# Long:  userID | assignment | totalCount | totalTime | grade | FinalScore
#
# This makes it much easier to merge with activity_summary,
# and also makes it easier to run the same analysis for all assignments at once.

# Step 1: Melt totalCount columns from wide to long
counts_long = grades.melt(
    id_vars='userID',
    value_vars=[f'a{i}totalCount' for i in range(1, 6)],
    var_name='assignment',
    value_name='totalCount'
)
# Extract just 'a1', 'a2', etc. from 'a1totalCount'
counts_long['assignment'] = counts_long['assignment'].str.extract(r'(a\d)')

# Step 2: Melt totalTime columns from wide to long
times_long = grades.melt(
    id_vars='userID',
    value_vars=[f'a{i}totalTime' for i in range(1, 6)],
    var_name='assignment',
    value_name='totalTime'
)
times_long['assignment'] = times_long['assignment'].str.extract(r'(a\d)')

# Step 3: Merge counts and times
activity_totals = pd.merge(counts_long, times_long, on=['userID', 'assignment'])

# Step 4: Look up the assignment grade (A1, A2, ...) for each row
assignment_to_grade_col = {f'a{i}': f'A{i}' for i in range(1, 6)}

activity_totals['grade'] = activity_totals.apply(
    lambda row: grades.loc[
        grades['userID'] == row['userID'],
        assignment_to_grade_col[row['assignment']]
    ].values[0],
    axis=1
)

# Step 5: Add the student's overall FinalScore
activity_totals = pd.merge(activity_totals, grades[['userID','FinalScore']], on='userID', how='left')

# Step 6: Clean up column order
activity_totals = activity_totals[['userID','assignment','grade','FinalScore','totalCount','totalTime']]

print(f"activity_totals shape: {activity_totals.shape}")
activity_totals.head()

Now merge the temporal features into our long-format dataset.

In [ ]:
# Merge activity_totals (counts + times + grades) with activity_summary (day features)
# Both are in long format keyed by (userID, assignment) → a simple left join works.
grades2 = pd.merge(activity_totals, activity_summary, on=['userID', 'assignment'], how='left')

print(f"grades2 shape: {grades2.shape}")
print("Columns:", grades2.columns.tolist())
grades2.head()

## Correlation matrix: all features vs. grades

Let's see how well our temporal features correlate with each assignment grade.

Because our data is now in long format, we first need to pivot it back to wide format
so that each column corresponds to one (feature, assignment) combination —
that's what we need for a heatmap.

In [ ]:
# ── Pivot long → wide for the correlation heatmap ───────────────────────────
#
# grades2.pivot(index='userID', columns='assignment') creates a wide DataFrame
# with MultiIndex columns like ('totalCount','a1'), ('totalCount','a2'), etc.
# We then flatten those into strings like 'totalCount_a1'.

grades_wide = grades2.pivot(index='userID', columns='assignment')
grades_wide.columns = [f"{col[0]}_{col[1]}" for col in grades_wide.columns]
grades_wide = grades_wide.reset_index()

# ── Set up the axes of the heatmap ───────────────────────────────────────────
grade_cols    = [f'grade_a{i}' for i in range(1, 6)]
assignments_  = ['a1', 'a2', 'a3', 'a4', 'a5']
feature_bases = ['totalCount','totalTime','firstDay','secondDay',
                 'firstSecondGap','nDays','nDaysFinalWeek',
                 'nDaysBeforeFinalWeek']

feature_cols  = [
    f"{feat}_{a}"
    for a in assignments_
    for feat in feature_bases
    if f"{feat}_{a}" in grades_wide.columns
]

# ── Compute same-assignment correlations only ─────────────────────────────────
# We only correlate a1 features with a1 grade, a2 features with a2 grade, etc.
# (Cross-assignment correlations would muddle interpretation at this stage.)

corr_matrix = pd.DataFrame(index=feature_cols, columns=grade_cols, dtype='float')

for a in assignments_:
    grade_col = f'grade_{a}'
    for feat in feature_bases:
        feat_col = f'{feat}_{a}'
        if grade_col in grades_wide.columns and feat_col in grades_wide.columns:
            corr_matrix.loc[feat_col, grade_col] = (
                grades_wide[[grade_col, feat_col]].corr().iloc[0, 1]
            )

corr_matrix = corr_matrix.astype(float)

# ── Plot ─────────────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 14))
sns.heatmap(
    corr_matrix, annot=True, cmap='coolwarm', center=0,
    fmt=".2f", vmin=-1, vmax=1
)
plt.title("Correlation — Activity Features vs. Assignment Grade", fontsize=13)
plt.xlabel("Grade Column")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("\n📌 Note: 'firstSecondGap' can be computed from firstDay and secondDay,")
print("   making it redundant (collinear). We'll drop it and 'nDaysBeforeFinalWeek' next.")

Some features are **redundant** (collinear): `firstSecondGap` can always be derived from `firstDay` and `secondDay`.
Keeping redundant features in a regression model can cause instability.
Let's drop them.

In [ ]:
# Drop redundant features
# - firstSecondGap  = firstDay - secondDay  →  redundant, derived from two other features
# - nDaysBeforeFinalWeek  →  partially captured by nDays and nDaysFinalWeek

grades2 = grades2.drop(columns=['firstSecondGap', 'nDaysBeforeFinalWeek'])

print("Remaining columns:")
grades2.info()

---
## Feature Family 3 — Ratios

Counts and totals answer *how much*. Ratios answer *what proportion*.

Consider:
- Student A spent 2 hours on the assignment and 1 hour on the detail spec page → **50% detail ratio**
- Student B spent 10 hours on the assignment and 1 hour on the detail spec page → **10% detail ratio**

Both spent the same absolute time on the detail page, but Student B spread their effort much more broadly.
A ratio feature captures this relative behaviour.

We will compute: **`detailRatio`** = (time spent on detail pages) / (total time on all assignment pages)

> 💡 A Canvas assignment typically has several pages. The **detail** page contains the full specification.
> Students who spend more relative time on the spec may be doing more careful reading.

In [ ]:
# ── Step 1: Filter log to only detail-page visits ────────────────────────────
#
# Detail page URLs match the pattern 'a1-detail', 'a2-detail', etc.
# We use a regular expression to select only those rows.

detail_pattern = r'a[1-5]-detail'
log_detail = log[log['url'].str.contains(detail_pattern, regex=True, na=False)]

print(f"Total log rows:        {len(log):,}")
print(f"Detail-page rows only: {len(log_detail):,}  ({100*len(log_detail)/len(log):.1f}%)")

# ── Step 2: Sum time on detail pages per (student, assignment) ────────────────
detail_time = (
    log_detail
    .groupby(['userID', 'assignment'])['view_length']
    .sum()
    .reset_index()
    .rename(columns={'view_length': 'detailTime'})
)

print(f"\ndetail_time shape: {detail_time.shape}")
detail_time.head()

In [ ]:
# ── Step 3: Merge detail time into grades2 ────────────────────────────────────
#
# A left join means students who never visited a detail page will have NaN
# for detailTime — we will fill those with 0 below.

grades2 = pd.merge(grades2, detail_time, on=['userID', 'assignment'], how='left')

# ── Step 4: Compute the ratio ─────────────────────────────────────────────────
#
# detailRatio = detailTime / totalTime
#
# Edge case: if totalTime == 0 (student never spent any time), the ratio would
# be 0/0 = NaN.  fillna(0) replaces those with 0.

grades2['detailRatio'] = grades2['detailTime'] / grades2['totalTime']
grades2['detailRatio'] = grades2['detailRatio'].fillna(0)

print("detailRatio distribution:")
print(grades2['detailRatio'].describe().round(3))

# Quick visualisation of the new ratio feature
plt.figure(figsize=(10, 4))
sns.boxplot(
    data=grades2, x='assignment', y='detailRatio',
    order=['a1','a2','a3','a4','a5'], palette='Set2'
)
plt.title('Distribution of Detail-Page Ratio per Assignment')
plt.xlabel('Assignment')
plt.ylabel('detailRatio  (0 = no time on detail page)')
plt.tight_layout()
plt.show()

Let's check whether `detailRatio` correlates with grades.

In [ ]:
# ── Correlation: detailRatio vs. grade ───────────────────────────────────────
#
# We loop over each assignment, filter to just those rows,
# and compute the Pearson correlation between detailRatio and grade.

correlations = {}
for assignment in sorted(grades2['assignment'].unique()):
    subset = grades2[grades2['assignment'] == assignment]
    if subset['detailRatio'].nunique() > 1:
        corr = subset[['grade', 'detailRatio']].corr().iloc[0, 1]
        correlations[assignment] = round(corr, 3)
    else:
        correlations[assignment] = None

corr_df = pd.DataFrame.from_dict(
    correlations, orient='index', columns=['Correlation with grade']
)
corr_df.index.name = 'Assignment'
print(corr_df)

print("\n📌 Positive correlations throughout — students who spend a higher proportion")
print("   of their time reading the spec tend to score better.")
print("   A2 and A5 are especially strong — worth investigating the assignment content!")

---
## Putting it all together — Linear Regression

Now that we have three families of features, let's fit a simple **multiple linear regression** model
to predict each assignment grade.

This is not the main focus of today's lecture, but it gives us a concrete way to evaluate
whether our feature engineering paid off:
- **R²** — what percentage of grade variance do our features explain?
- **RMSE** — on average, how many grade points is our prediction off by?
- **Coefficients** — which features are most influential?

> ⚠️ **Important:** We are fitting on the full dataset (no train/test split) just to get a feel for the features.
> In a proper study you would use cross-validation (see the `LeaveOneOut` import below).

In [ ]:
from sklearn.model_selection import LeaveOneOut, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# ── Helper function: fit a linear regression and print a readable summary ─────
def run_LR(X, y, predictors):
    """
    Fit a LinearRegression model, print the regression formula,
    and report RMSE and R² on the training data.
    
    Parameters
    ----------
    X          : pd.DataFrame of predictor features
    y          : pd.Series of the target variable (grade)
    predictors : list of feature names (for printing the formula)
    """
    model = LinearRegression()
    model.fit(X, y)
    
    # Print the regression formula: y = intercept + b1*x1 + b2*x2 + ...
    print(f"Regression formula:  grade = {model.intercept_:.2f}")
    for coef, name in zip(model.coef_, predictors):
        sign = '+' if coef >= 0 else '-'
        print(f"               {sign} {abs(coef):.4f} × {name}")
    
    y_pred    = model.predict(X)
    rmse      = np.sqrt(np.mean((y - y_pred) ** 2))
    r_squared = model.score(X, y)
    print(f"  RMSE:      {rmse:.1f} grade points")
    print(f"  R²:        {r_squared:.3f}  ({r_squared*100:.1f}% of variance explained)")

Let's run a separate model for each assignment.

Note: `totalTime` is in **seconds**, which is a very large number compared to a grade (0–100).
The coefficient printed will look like 0.000, not because time has no effect,
but because one extra *second* barely moves the grade.
We'll convert to hours to make the coefficient interpretable.

In [ ]:
# ── Feature list for the regression ──────────────────────────────────────────
feature_cols = ['totalCount', 'totalTime', 'firstDay', 'secondDay',
                'nDays', 'nDaysFinalWeek', 'nDaysAfterDeadline', 'detailRatio']

# Convert totalTime from seconds to hours so the coefficient is easier to read
# (e.g., "1 extra hour predicts +2.3 grade points" is more meaningful than
#  "3600 extra seconds predicts +2.3 grade points")
grades3 = grades2.copy()
grades3['totalTime'] = grades3['totalTime'] / 3600   # seconds → hours

print(f"Features used: {feature_cols}\n")

for a in ['a1', 'a2', 'a3', 'a4', 'a5']:
    subset = grades3[grades3['assignment'] == a].dropna(subset=feature_cols + ['grade'])
    X = subset[feature_cols]
    y = subset['grade']
    
    if len(subset) > len(feature_cols):
        print(f"{'='*55}")
        print(f"Assignment {a.upper()}  ({len(subset)} students)")
        print(f"{'='*55}")
        run_LR(X, y, feature_cols)
    else:
        print(f"Skipping {a}: not enough data ({len(subset)} rows).")
    print()

### Interpreting the model

**How to read the coefficients:**
- A coefficient of `+2.5` on `firstDay` means: *for every extra day earlier a student started, the model predicts 2.5 more grade points* (all else being equal).
- A coefficient of `−5.6` on `nDaysAfterDeadline` means: *each day a student accessed pages after the deadline is associated with −5.6 grade points* — possibly indicating late or incomplete submissions.

**Why R² is modest (~0.25):**
> This model has no information about the student's background or programming skill.
> A strong programmer might spend very little time on Canvas yet score 95.
> A struggling student might spend many hours yet score 40.
> Our behavioural features alone can explain about 25% of the variance — which is actually meaningful given what we're working with!

---
## Summary: What we did today

| Step | Action | Pandas / sklearn tool |
|---|---|---|
| 1 | Counted page views per student per assignment | `groupby().size().unstack()` |
| 2 | Summed time spent per student per assignment | `groupby()[col].sum().unstack()` |
| 3 | Computed temporal features (days before deadline) | Manual loop + `pd.DataFrame` |
| 4 | Reshaped data from wide to long | `pd.melt()` + `pd.merge()` |
| 5 | Computed ratio feature (detail page proportion) | Column arithmetic |
| 6 | Evaluated all features with correlation heatmap | `.corr()` + `sns.heatmap()` |
| 7 | Dropped redundant/collinear features | `df.drop(columns=[...])` |
| 8 | Fit linear regression to assess predictive power | `sklearn.LinearRegression` |

**Key takeaways:**
- Feature engineering turns raw events into model-ready inputs using domain knowledge.
- Always **visualise and check correlations** before using features in a model.
- **Redundant features** (those that can be derived from others) should be removed to avoid collinearity.
- **Scale matters**: a feature in seconds and a feature in counts are on very different scales — consider normalising or converting units before interpreting coefficients.